# Deployment do Modelo de Produção via MLflow Model Serving

Este notebook demonstra como consumir o modelo de Produção registrado no MLflow Model Registry usando a API de Model Serving do Databricks.

* O modelo foi registrado como `heart-disease-model` com alias `Production`.
* O endpoint de inferência permite enviar dados raw e receber predições diretamente.

## Load Endpoint Configuration

In [1]:
import yaml

model_name = 'workspace.default.heart-disease-model'
model_version = 7

# Load endpoint configuration
with open("endpoint.yaml", "r") as f:
  conf = yaml.safe_load(f)
  endpoint_config = conf['endpoint_config']
  endpoint_config['config']['served_entities'][0]['entity_name'] = model_name
  endpoint_config['config']['served_entities'][0]['entity_version'] = model_version

  display(endpoint_config)

{'config': {'served_entities': [{'name': 'heart-disease-model',
    'workload_type': 'CPU',
    'workload_size': 'Small',
    'scale_to_zero_enabled': True,
    'entity_name': 'workspace.default.heart-disease-model',
    'entity_version': 7}],
  'traffic_config': {'routes': [{'served_model_name': 'heart-disease-model',
     'traffic_percentage': 100}]}},
 'tags': [{'key': 'project', 'value': 'heart-disease'}]}

## Create/Update Endpoint

In [0]:
import mlflow
from mlflow.deployments import get_deploy_client

endpoint_name = 'heart-disease-predict'

# Initialize Mlflow client
mlflow.set_registry_uri("databricks-uc")
client = get_deploy_client("databricks")

# Verify if endpoint exists
try:
  client.get_endpoint(endpoint_name)
  endpoint_exists = True
  print(f'Endpoint {endpoint_name} already exists...')
except:
  endpoint_exists = False
  print(f'Endpoint {endpoint_name} does not exist. Creating new endpoint...')

In [0]:

# Create endpoint
if endpoint_exists:
    # Update endpoint
    print(f'Updating endpoint {endpoint_name}...')
    endpoint = client.update_endpoint_config(
        endpoint=endpoint_name,
        config=endpoint_config['config']
    )
else:   
    print(f'Creating endpoint {endpoint_name}...')
    endpoint = client.create_endpoint(
        name=endpoint_name,
        config=endpoint_config,
    )

In [0]:
# Exemplo: enviar dados raw para o endpoint de Model Serving
import pandas as pd
import os
import requests
import json

TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# URL do endpoint de Model Serving (ajuste conforme necessário)
ENDPOINT_URL = "https://<YOUR-WORKSPACE>.cloud.databricks.com/serving-endpoints/heart-disease-predict/invocations"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# Carregar amostra de dados raw
raw_data_path = "../../data/heart_disease_uci.csv"
df_raw = pd.read_csv(raw_data_path)

# Remover colunas de metadados conforme pipeline
columns_to_drop = ['id', 'dataset', 'num', 'target']
sample = df_raw.iloc[[0]].copy()
for col in columns_to_drop:
    if col in sample.columns:
        sample = sample.drop(columns=col)

# Preparar payload para API
payload = {
    "dataframe_split": {
        "columns": sample.columns.tolist(),
        "data": sample.values.tolist()
    }
}

print(json.dumps(payload))

# Enviar requisição ao endpoint
response = requests.post(ENDPOINT_URL, headers=headers, data=json.dumps(payload))
print(f"Status code: {response.status_code}")
print(f"Resposta: {response.json()}")

## Boas Práticas para Deployment

* **Consistência**: Use o mesmo pipeline de pré-processamento do treino para inferência.
* **Segurança**: Proteja o endpoint com autenticação via token.
* **Monitoramento**: Acompanhe métricas de uso e performance do endpoint.
* **Versionamento**: Utilize aliases (`Production`, `Staging`) para gerenciar versões do modelo.
* **Validação**: Sempre valide o formato dos dados enviados ao endpoint.

---